# Testing QuaC on real datasets


In [ ]:
import qiskit
from qiskit_ibm_runtime import QiskitRuntimeService
import numpy as np
from collections import Counter
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister
import time
from itertools import combinations
import csv


### We will read each line and encode it 

In [ ]:
x = {}
fx = {}
pairs = {}

x_counter = 0
fx_counter = 0

start_time = time.time()

with open('annotation_message.csv', mode ='r')as file:
  csvFile = csv.reader(file)
  for lines in csvFile:
    # Checking if x already exists
    if lines[0] not in x:
        x[lines[0]] = x_counter
        x_counter += 1
    else:
       print(
            f"Duplicate x value. Not functional. "
            f"{lines[0]} has been encountered before in the dataset."
        )
       continue

    # Checking if fx already exists
    #fx_item = lines[1] + " " + lines[2]
    fx_item = lines[1]
    if fx_item not in fx:
        fx[fx_item] = fx_counter
        fx_counter += 1
    else:
       print("Duplicate f(x)")
    pairs[x[lines[0]]] = (fx[fx_item] + 11) 
end_time = time.time()
       

elapsed = end_time - start_time
print(f"\n⏱️ Encoding time: {elapsed:.4f} seconds")


### Randomized encoding (Optional. Used if the function is 1:1 and indexing is not required)

In [ ]:
x = {}
fx = {}
rows = []

x_counter = 0
fx_counter = 0

with open('catalog_url.csv', mode='r') as file:
    csvFile = csv.reader(file)
    for lines in csvFile:
        x_item = lines[0]
        fx_item = lines[1]

        if x_item in x:
            print(f"Duplicate x value: {x_item}")
            continue

        x[x_item] = x_counter
        x_counter += 1

        if fx_item not in fx:
            fx[fx_item] = fx_counter
            fx_counter += 1

        rows.append((x_item, fx_item))

import random

# Create list of all possible output IDs
perm = list(range(fx_counter))

# Shuffle it
random.shuffle(perm)

pairs = {}

for x_item, fx_item in rows:
    xi = x[x_item]
    yi = fx[fx_item]

    pairs[xi] = perm[yi]

### Now we need to express these pairs in 0 and 1 with the minimum numbers of bits

In [ ]:
max_x = max(pairs)
max_fx = max(pairs.values())
print(max_x)
x_bits = len(bin(max_x)) - 2
fx_bits = len(bin(max_fx)) - 2

print(x_bits)
print(fx_bits)

pairs_binary = {}

for pair in pairs:
    pairs_binary[format(pair,f'0{x_bits}b')] = format(pairs[pair],f'0{fx_bits}b')

#print(pairs_binary)

### Now we need to prepare the input for the algorithm execution

In [ ]:
# Now we will prepare the input for the Younes-Miller algorithm
# We need to collect in a list, each input (ex. 101101) that makes the selected bit of output 1 (ex 101011<- for the first bit)
# We also need to organize them into lists and tuples to match the expected input of the algorithm implementation

inputs = [[] for _ in range(fx_bits)]
start_time = time.time()
# For each different input in our pair dictionary, we will add it in the nth output bit list only if that bit is 1
for pair in pairs_binary:
    inputTuple = tuple(int(bit) for bit in pair)
    for i,bit in enumerate(pairs_binary[pair]):
        if bit == '1':
            inputs[i].append(inputTuple)

end_time = time.time()

# Display elapsed time
elapsed = end_time - start_time
print(f"\n⏱️ Preparing input time: {elapsed:.4f} seconds")

#print(inputs)


# Running the algorithm

In [ ]:





gates = 0


def minimizeAndGenerate(outputCounter,all0): # Function to minimize the number of AND gates and add them to the circuit
    visited = [] # List to store unique control combinations
    unique = []
    global gates

    counter = Counter(tuple(c) for c in controlsArray) 
    unique = [list(k) for k, v in counter.items() if v % 2 != 0] #Keeping only the gates that appear odd number of times since the even ones cancel out

    for u in unique:
        controls = [a[j] for j in range(inputSize) if u[j] == 1] # Getting the control qubits from the unique combinations
        if len(controls)!=0:
            qc.mcx(controls, b[outputCounter]) # Adding the controls to the circuit
            gates += 1
    if all0 == 1:
        qc.x(b[outputCounter]) # Adding the NOT gate if required
        gates += 1
    qc.measure(b[outputCounter], outputCounter) # Measuring the output
    controlsArray.clear() # Resetting the controls array for the next input
    unique.clear() # Resetting the unique list for the next input


start_time = time.time()
inputSize = x_bits
outputSize = fx_bits
conf = inputs
gates = 0

a = QuantumRegister(inputSize, 'a')
b = QuantumRegister(outputSize, 'b')
qc = QuantumCircuit(a, b)
c = ClassicalRegister(outputSize, 'output_b0')
qc.add_register(c)

all0 = 0 
cond0 = []
controlsArray = []
outputCounter = 0

for c in conf:
    for input in c:
        cond0= []
        tmp = 0
        for i in range(inputSize):
            if input[i] == 0:
                cond0.insert(0,i)   # Storing the cond0 qubits
            else:
                tmp = 1
        if(tmp==0):
            all0 = 1                        
        for i in range(0, len(cond0)+1):
                for combo in combinations(cond0, i):
                    column = [1] * len(input)                   # Initializing a column with all 1s
                    for j in combo:
                        column[j] = 0                   # Setting the cond0 qubits to 0
                    controlsArray.append(column)                # Appending the column to the controls array
    
    minimizeAndGenerate(outputCounter,all0) # Calling the minimizer function that will also add the gates to the circuit
    all0 = 0 # Resetting the all 0 flag for the next input set
    if(outputSize > 1): 
        outputCounter += 1


end_time = time.time()

# Display elapsed time
elapsed = end_time - start_time
print(f"\n⏱️ Execution time: {elapsed:.4f} seconds")
print(f"\n Gates needed: {gates}")
print(f"\n Circuit depth: {qc.depth()}")

#qc.draw(output='mpl')  


In [ ]:



qc.measure_all()

SUPPORTED_MCX = {
    "x", "cx", "ccx", "mcx", "mcx_gray", "mcx_recursive", "mcx_vchain"
}

def run_classically_io(qc: QuantumCircuit, inputs, outputs, return_counts=False):
    if len(qc.qregs) != 2:
        raise ValueError(
            f"Expected 2 quantum registers (inputs/outputs), got {len(qc.qregs)}."
        )

    qin, qout = qc.qregs
    n_in, n_out = len(qin), len(qout)

    if len(inputs) != n_in or len(outputs) != n_out:
        raise ValueError(
            f"Input/output lengths must match register sizes "
            f"({n_in} / {n_out}), got ({len(inputs)} / {len(outputs)})."
        )

    # dataset, bits, rows, encode dataset, circuit time, circuit gates, compute fx, decode time
    bits = inputs + outputs
    qindex = {q: i for i, q in enumerate(qc.qubits)}
    creg_values = {} 

    # --- simulate -------------------------------------------------------------
    for instr in qc.data:
        op = instr.operation
        name = op.name

        # Skip barriers and delays
        if name in {"barrier", "id", "delay"}:
            continue

        # Measurement handling
        if name == "measure":
            q = instr.qubits[0]
            c = instr.clbits[0]
            creg_values[c] = bits[qindex[q]]
            continue

        # Logic gates
        if name not in SUPPORTED_MCX:
            raise ValueError(f"Unsupported gate: {name}")

        qubits = [qindex[q] for q in instr.qubits]

        if name == "x":
            bits[qubits[0]] ^= 1
        else:
            controls, target = qubits[:-1], qubits[-1]
            if all(bits[c] for c in controls):
                bits[target] ^= 1

    # --- split and optional counts --------------------------------------------
    final_inputs = bits[:n_in]
    final_outputs = bits[n_in:]

    if return_counts:
        bitstring = ''.join(str(b) for b in reversed(final_outputs))
        return (final_inputs, final_outputs), {bitstring: 1}
    else:
        return final_inputs, final_outputs


#

inputs  = [0,0,0,0,0,0,0,0,0,0,0,0,0,0]
outputs = [0]*fx_bits

start_time = time.time()




final_inputs, final_outputs = run_classically_io(qc, inputs, outputs)

print("Final inputs :", final_inputs)
print("Final outputs:", final_outputs)
end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱️ Output of the quantum circuit: {elapsed:.4f} seconds")

start_time = time.time()
decode_x = int(''.join(str(dec_x) for dec_x in final_inputs),2)
decode_fx = int(''.join(str(dec_fx) for dec_fx in final_outputs),2)

print("Input: "+ list(x.keys())[list(x.values()).index(decode_x)])
print("Output: "+ list(fx.keys())[list(fx.values()).index(decode_fx)])

end_time = time.time()
elapsed = end_time - start_time
print(f"\n⏱️ Decode output: {elapsed:.4f} seconds")